# FraudSpotter: Explainable Fake News Classification
## Google Colab T4 GPU Execution Notebook (Connected via Antigravity Colab Extension)

This notebook executes all GPU-intensive tasks directly on a connected Google Colab T4 runtime:
1. **GPU & Google Drive Verification**: Verifies NVIDIA T4 GPU and mounts Google Drive for persistent artifact storage.
2. **Repository Sync & Environment Setup**: Clones/updates repo to `/content/fraudspotter`, installs dependencies from `requirements-colab.txt`, and registers `fraudspotter` in editable mode.
3. **Dataset Verification**: Validates the frozen WELFake Parquet splits (Train: 50,937, Val: 6,373, REAL=0, FAKE=1) in Google Drive.
4. **DistilBERT Fine-Tuning**: Trains DistilBERT across sequence lengths (128, 256, 384) with paired `[CLS] Title [SEP] Body [SEP]` inputs, FP16 precision, and Macro-F1 checkpoint selection.
5. **Validation Benchmark Summary**: Reports final validation Macro-F1, Accuracy, Precision, Recall, MCC, and ROC-AUC for all sequence lengths.

### Step 1: GPU Verification & Google Drive Mount

In [ ]:
# 1. Verify T4 GPU access
!nvidia-smi

# 2. Mount Google Drive idempotently
import os
from google.colab import drive

if not os.path.exists("/content/drive/MyDrive"):
    print("Mounting Google Drive at /content/drive...")
    drive.mount('/content/drive')
else:
    print("[PASS] Google Drive is already mounted at /content/drive.")

DRIVE_BASE = "/content/drive/MyDrive/FraudSpotter"
DRIVE_DATA = f"{DRIVE_BASE}/data/processed/welfake_v1"
DRIVE_MODELS = f"{DRIVE_BASE}/models"

os.makedirs(DRIVE_MODELS, exist_ok=True)
print(f"[PASS] Persistent Google Drive storage verified at: {DRIVE_BASE}")

### Step 2: Sync Repository & Install Requirements
Pulls the latest codebase from GitHub, configures `sys.path`, and installs `fraudspotter` and GPU dependencies.

In [ ]:
import os
import sys

# Ensure Hugging Face Transformers uses pure PyTorch backend (avoids TF conflict in Colab)
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["USE_FLAX"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"

REPO_URL = "https://github.com/krish10a/FraudSpotter-Fake-News-Classification-Using-Transformer-Based-Deep-Learning.git"
WORKSPACE_DIR = "/content/fraudspotter"

if not os.path.exists(WORKSPACE_DIR):
    print("Cloning repository to /content/fraudspotter...")
    !git clone {REPO_URL} {WORKSPACE_DIR}
else:
    print("Updating existing repository at /content/fraudspotter...")
    %cd {WORKSPACE_DIR}
    !git fetch origin main
    !git reset --hard origin/main

%cd {WORKSPACE_DIR}

# Ensure workspace and src are in sys.path
for p in [WORKSPACE_DIR, os.path.join(WORKSPACE_DIR, "src")]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Install GPU dependencies and package in editable mode
!pip install --upgrade pip -q
!pip install -r requirements-colab.txt -q
!pip install -e . -q

# Verify fraudspotter import
import fraudspotter
print(f"\n[PASS] fraudspotter library loaded successfully from: {fraudspotter.__file__}")

### Step 3: Dataset Ingestion Verification
Locates and validates the frozen WELFake split v1 (50,937 train, 6,373 val) in Google Drive.

In [ ]:
import os
from pathlib import Path
import pandas as pd

# Search possible locations in Google Drive
candidate_dirs = [
    Path(DRIVE_DATA),
    Path("/content/drive/MyDrive/FraudSpotter/data/processed/welfake_v1"),
    Path("/content/drive/MyDrive/FraudSpotter_Data"),
    Path("/content/drive/MyDrive/welfake_v1"),
    Path("/content/drive/MyDrive/FraudSpotter"),
]

DATA_DIR = None
for c_dir in candidate_dirs:
    if (c_dir / "train.parquet").exists() and (c_dir / "validation.parquet").exists():
        DATA_DIR = c_dir
        break

# Fallback: scan Google Drive if not in default paths
if DATA_DIR is None and Path("/content/drive/MyDrive").exists():
    print("Searching Google Drive for frozen Parquet splits...")
    for root, dirs, files in os.walk("/content/drive/MyDrive"):
        if "train.parquet" in files and "validation.parquet" in files:
            DATA_DIR = Path(root)
            print(f"Discovered dataset in: {DATA_DIR}")
            break

if DATA_DIR is not None:
    train_parquet = DATA_DIR / "train.parquet"
    val_parquet = DATA_DIR / "validation.parquet"
    tr_df = pd.read_parquet(train_parquet, columns=["label"])
    vl_df = pd.read_parquet(val_parquet, columns=["label"])
    print(f"\n[PASS] Frozen dataset splits verified:")
    print(f"       - Source directory:     {DATA_DIR}")
    print(f"       - Train partition:      {len(tr_df):,} records (REAL={sum(tr_df['label']==0):,}, FAKE={sum(tr_df['label']==1):,})")
    print(f"       - Validation partition: {len(vl_df):,} records (REAL={sum(vl_df['label']==0):,}, FAKE={sum(vl_df['label']==1):,})")
else:
    print("\n[WARN] Frozen Parquet splits not found in Drive. Running automated remote download & preprocessing...")
    import urllib.request
    raw_dir = Path("/content/fraudspotter/data/raw")
    raw_dir.mkdir(parents=True, exist_ok=True)
    raw_file = raw_dir / "welfake.csv"
    if not raw_file.exists():
        print("Downloading WELFake from Zenodo (~110MB)..."
        zenodo_url = "https://zenodo.org/record/4561253/files/WELFake_Dataset.csv?download=1"
        urllib.request.urlretrieve(zenodo_url, str(raw_file))
    os.makedirs(DRIVE_DATA, exist_ok=True)
    !python scripts/preprocess_data.py --input_file /content/fraudspotter/data/raw/welfake.csv --output_dir {DRIVE_DATA}
    DATA_DIR = Path(DRIVE_DATA)
    print(f"[PASS] Remote preprocessing complete. Splits created at: {DATA_DIR}")

### Step 4: DistilBERT Fine-Tuning across Sequence Lengths (128, 256, 384)
Executes fine-tuning for all 3 sequence length configurations:
- **Sequence Lengths**: `128`, `256` (canonical), `384`
- **Paired Input**: `[CLS] Title [SEP] Body [SEP]`
- **Precision**: FP16 mixed precision
- **Batch Size & Accumulation**:
  - `seq_len=128`: Batch 32, Grad Accum 1
  - `seq_len=256`: Batch 16, Grad Accum 1
  - `seq_len=384`: Batch 8, Grad Accum 2 (avoids T4 16GB VRAM OOM)
- **Selection**: Best model saved by validation Macro-F1
- **Storage**: Checkpoints saved directly to Google Drive

In [ ]:
import subprocess
import sys
import os
import json
from pathlib import Path

train_path = DATA_DIR / "train.parquet" if (DATA_DIR / "train.parquet").exists() else DATA_DIR / "train.csv"
val_path = DATA_DIR / "validation.parquet" if (DATA_DIR / "validation.parquet").exists() else DATA_DIR / "val.csv"

# Sequence length experiments: 128, 256 (canonical), 384
seq_lengths = [128, 256, 384]
results_summary = {}

for s_len in seq_lengths:
    print(f"\n{'=' * 65}")
    print(f" Starting DistilBERT Fine-Tuning: Sequence Length = {s_len}")
    print(f"{'=' * 65}")
    
    model_out = f"{DRIVE_MODELS}/distilbert_seq{s_len}"
    os.makedirs(model_out, exist_ok=True)
    
    cmd = [
        sys.executable,
        "scripts/train_transformer.py",
        "--train_file", str(train_path),
        "--val_file", str(val_path),
        "--seq_len", str(s_len),
        "--output_dir", str(model_out),
    ]
    
    print("Executing command:", " ".join(cmd))
    env_vars = os.environ.copy()
    env_vars["USE_TF"] = "0"
    env_vars["USE_TORCH"] = "1"
    env_vars["USE_FLAX"] = "0"
    env_vars["TRANSFORMERS_NO_TF"] = "1"
    env_vars["TRANSFORMERS_NO_FLAX"] = "1"
    env_vars["PYTHONPATH"] = f"/content/fraudspotter:/content/fraudspotter/src:{env_vars.get('PYTHONPATH', '')}"
    
    # Real-time streaming subprocess
    process = subprocess.Popen(
        cmd,
        cwd="/content/fraudspotter",
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env_vars,
    )
    
    for line in iter(process.stdout.readline, ''):
        print(line, end='')
    process.stdout.close()
    return_code = process.wait()
    
    if return_code != 0:
        print(f"\n[FAIL] Training failed for seq_len={s_len} with return code {return_code}")
        break
    
    # Load validation metrics
    metrics_path = Path(f"{model_out}/validation_metrics_seq{s_len}.json")
    if metrics_path.exists():
        with open(metrics_path, "r", encoding="utf-8") as f:
            results_summary[f"distilbert_seq{s_len}"] = json.load(f)
        print(f"\n[SUCCESS] Completed seq_len={s_len}. Metrics saved to {metrics_path}")

print("\n" + "=" * 70)
print(" DISTILBERT VALIDATION BENCHMARK SUMMARY (T4 GPU)")
print("=" * 70)
header = f"{'Configuration':<22} | {'Macro-F1':<9} | {'Accuracy':<9} | {'ROC-AUC':<9} | {'Fake-F1':<9} | {'Real-F1':<9}"
print(header)
print("-" * len(header))
for exp_name, exp_data in results_summary.items():
    v = exp_data.get("validation_metrics", {})
    macro_f1 = v.get("eval_f1_macro", v.get("eval_eval_f1_macro", 0.0))
    acc = v.get("eval_accuracy", v.get("accuracy", 0.0))
    roc_auc = v.get("eval_roc_auc", v.get("roc_auc", 0.0))
    fake_f1 = v.get("eval_fake_f1", v.get("fake_f1", 0.0))
    real_f1 = v.get("eval_real_f1", v.get("real_f1", 0.0))
    print(f"{exp_name:<22} | {macro_f1:<9.4f} | {acc:<9.4f} | {roc_auc:<9.4f} | {fake_f1:<9.4f} | {real_f1:<9.4f}")
print("=" * 70)

### Step 5: Checkpoint & Metric Verification
Audits the persisted best model directories and evaluation JSON files in Google Drive.

In [ ]:
from pathlib import Path
import json

print("=== Persisted Model Checkpoints in Google Drive ===")
for s_len in [128, 256, 384]:
    best_dir = Path(f"{DRIVE_MODELS}/distilbert_seq{s_len}/best_model")
    metrics_file = Path(f"{DRIVE_MODELS}/distilbert_seq{s_len}/validation_metrics_seq{s_len}.json")
    status_model = "[FOUND]" if best_dir.exists() else "[MISSING]"
    status_metrics = "[FOUND]" if metrics_file.exists() else "[MISSING]"
    print(f"seq_len={s_len:<3} -> Best Model: {status_model:<9} | Metrics: {status_metrics}")
    if metrics_file.exists():
        with open(metrics_file, "r", encoding="utf-8") as f:
            data = json.load(f)
            v = data.get("validation_metrics", {})
            print(f"             Macro-F1: {v.get('eval_f1_macro', 0.0):.4f} | Accuracy: {v.get('eval_accuracy', 0.0):.4f}")
print("\n[COMPLETE] All DistilBERT checkpoints verified in Google Drive.")